# ⚙ Notebook 10: Hyperparameter Tuning

---

**Author:** Dnyanesh  
**Date:** February 2025  
**Series:** Tata Motors Deep Dive (10 of 13)

## Objective

Default hyperparameters rarely give the best performance. We use systematic search to find optimal parameters for:
1. **Random Forest** — via Optuna/GridSearch
2. **XGBoost** — via Optuna/GridSearch

### Why Tune?
| Default | Tuned |
|---------|-------|
| One-size-fits-all | Optimized for THIS dataset |
| May overfit or underfit | Balanced bias-variance |
| Quick to set up | Better generalization |

In [ ]:
# ============================================================
# Setup
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, make_scorer
import os, warnings, time
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
PROCESSED_DIR = '../data/processed'
print('✅ Environment ready')

In [ ]:
# Optional libraries
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except: OPTUNA_OK = False
try:
    from xgboost import XGBClassifier
    XGB_OK = True
except: XGB_OK = False
try:
    from lightgbm import LGBMClassifier
    LGBM_OK = True
except: LGBM_OK = False
print(f'Optuna:   {"✅" if OPTUNA_OK else "❌"}')
print(f'XGBoost:  {"✅" if XGB_OK else "❌"}')
print(f'LightGBM: {"✅" if LGBM_OK else "❌"}')

In [ ]:
# ============================================================
# Load data
# ============================================================
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'tata_motors_all_features.csv'), index_col=0, parse_dates=True) if os.path.exists(os.path.join(PROCESSED_DIR, 'tata_motors_all_features.csv')) else pd.read_csv(os.path.join(PROCESSED_DIR, 'tata_motors_clean.csv'), index_col=0, parse_dates=True)
if 'Log_Return' not in df.columns:
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
df['Target'] = (df['Log_Return'].shift(-1) > 0).astype(int)

sel_file = os.path.join(PROCESSED_DIR, 'selected_features.csv')
if os.path.exists(sel_file):
    selected = [f for f in pd.read_csv(sel_file).iloc[:, 0].tolist() if f in df.columns]
else:
    exclude = ['Target', 'Regime', 'Cluster', 'Log_Return', 'Simple_Return', 'Price_Change', 'Gain', 'Loss', 'Avg_Gain', 'Avg_Loss', 'RS']
    selected = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude][:15]

model_df = df[selected + ['Target']].dropna()
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(model_df[selected]), columns=selected, index=model_df.index)
y = model_df['Target']
tscv = TimeSeriesSplit(n_splits=5)
print(f'Features: {len(selected)}, Samples: {len(X)}')

---

## Part 1: Understanding Hyperparameters

### Random Forest Key Parameters:

| Parameter | Range | Effect |
|-----------|-------|--------|
| `n_estimators` | 50-500 | More trees = better but slower |
| `max_depth` | 3-20 | Deeper = more complex |
| `min_samples_split` | 2-20 | Higher = more conservative |
| `min_samples_leaf` | 1-10 | Prevents small leaf nodes |
| `max_features` | sqrt, log2, None | Controls randomness |

### XGBoost Key Parameters:

| Parameter | Range | Effect |
|-----------|-------|--------|
| `learning_rate` | 0.01-0.3 | Step size (smaller = careful) |
| `max_depth` | 3-12 | Tree complexity |
| `subsample` | 0.6-1.0 | Row sampling |
| `colsample_bytree` | 0.6-1.0 | Column sampling |
| `reg_alpha` | 0-10 | L1 regularization |
| `reg_lambda` | 0-10 | L2 regularization |

In [ ]:
# ============================================================
# 1.1 Default performance baseline
# ============================================================
rf_default = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_score = cross_val_score(rf_default, X, y, cv=tscv, scoring='accuracy')

print('DEFAULT PERFORMANCE BASELINE')
print('=' * 50)
print(f'Random Forest (default):')
print(f'  Fold scores: {[f"{s:.4f}" for s in rf_score]}')
print(f'  Mean: {rf_score.mean():.4f} ± {rf_score.std():.4f}')

if XGB_OK:
    xgb_default = XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss')
    xgb_score = cross_val_score(xgb_default, X, y, cv=tscv, scoring='accuracy')
    print(f'\nXGBoost (default):')
    print(f'  Fold scores: {[f"{s:.4f}" for s in xgb_score]}')
    print(f'  Mean: {xgb_score.mean():.4f} ± {xgb_score.std():.4f}')

---

## Part 2: Random Forest Tuning

### 2.1 Approach
We use either Optuna (if available) or GridSearchCV as fallback.

In [ ]:
# ============================================================
# 2.1 Random Forest — Optuna / Grid Search
# ============================================================
start_time = time.time()

if OPTUNA_OK:
    def rf_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'random_state': 42, 'n_jobs': -1
        }
        model = RandomForestClassifier(**params)
        scores = cross_val_score(model, X, y, cv=tscv, scoring='accuracy')
        return scores.mean()
    
    study_rf = optuna.create_study(direction='maximize')
    study_rf.optimize(rf_objective, n_trials=50, show_progress_bar=False)
    rf_best_score = study_rf.best_value
    rf_best_params = study_rf.best_params
else:
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10]
    }
    grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                       param_grid, cv=tscv, scoring='accuracy', n_jobs=-1)
    grid.fit(X, y)
    rf_best_score = grid.best_score_
    rf_best_params = grid.best_params_

rf_time = time.time() - start_time
print('RANDOM FOREST TUNING RESULTS')
print('=' * 50)
print(f'Best accuracy: {rf_best_score:.4f}')
print(f'Improvement:   {(rf_best_score - rf_score.mean())*100:+.2f}%')
print(f'Time taken:    {rf_time:.1f}s')
print(f'Best params:   {rf_best_params}')

In [ ]:
# ============================================================
# 2.2 RF Optimization History
# ============================================================
if OPTUNA_OK:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    ax = axes[0]
    vals = [t.value for t in study_rf.trials]
    ax.plot(vals, 'o-', color='#3498DB', alpha=0.5, markersize=4)
    ax.axhline(study_rf.best_value, color='red', linestyle='--', label=f'Best: {study_rf.best_value:.4f}')
    ax.axhline(rf_score.mean(), color='gray', linestyle=':', label=f'Default: {rf_score.mean():.4f}')
    ax.set_title('RF Optimization History', fontweight='bold')
    ax.set_xlabel('Trial'); ax.set_ylabel('Accuracy'); ax.legend()
    
    ax = axes[1]
    importances = optuna.importance.get_param_importances(study_rf)
    ax.barh(list(importances.keys()), list(importances.values()), color='#2ECC71', alpha=0.7)
    ax.set_title('RF Parameter Importance', fontweight='bold')
    ax.set_xlabel('Importance')
    
    plt.tight_layout(); plt.show()
else:
    print('GridSearchCV used — no optimization history plot')

---

## Part 3: XGBoost Tuning

### XGBoost's Unique Parameters

XGBoost adds **regularization** to gradient boosting:

$$\text{Objective} = \sum_{i} L(y_i, \hat{y}_i) + \sum_{k} \Omega(f_k)$$

where $\Omega(f) = \gamma T + \frac{1}{2}\lambda ||w||^2 + \alpha||w||_1$

- $\gamma$: Minimum loss reduction for split
- $\lambda$: L2 regularization (reg_lambda)
- $\alpha$: L1 regularization (reg_alpha)

In [ ]:
# ============================================================
# 3.1 XGBoost Tuning
# ============================================================
if XGB_OK:
    start_time = time.time()
    
    if OPTUNA_OK:
        def xgb_objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
                'random_state': 42, 'use_label_encoder': False,
                'eval_metric': 'logloss', 'verbosity': 0
            }
            model = XGBClassifier(**params)
            scores = cross_val_score(model, X, y, cv=tscv, scoring='accuracy')
            return scores.mean()
        
        study_xgb = optuna.create_study(direction='maximize')
        study_xgb.optimize(xgb_objective, n_trials=50, show_progress_bar=False)
        xgb_best_score = study_xgb.best_value
        xgb_best_params = study_xgb.best_params
    else:
        param_grid = {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7]
        }
        grid = GridSearchCV(XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'),
                           param_grid, cv=tscv, scoring='accuracy', n_jobs=-1)
        grid.fit(X, y)
        xgb_best_score = grid.best_score_
        xgb_best_params = grid.best_params_
    
    xgb_time = time.time() - start_time
    print('\nXGBOOST TUNING RESULTS')
    print('=' * 50)
    print(f'Best accuracy: {xgb_best_score:.4f}')
    print(f'Improvement:   {(xgb_best_score - xgb_score.mean())*100:+.2f}%')
    print(f'Time taken:    {xgb_time:.1f}s')
    print(f'Best params:   {xgb_best_params}')

In [ ]:
# ============================================================
# 3.2 XGBoost Optimization History
# ============================================================
if XGB_OK and OPTUNA_OK:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    ax = axes[0]
    vals = [t.value for t in study_xgb.trials]
    ax.plot(vals, 'o-', color='#E74C3C', alpha=0.5, markersize=4)
    ax.axhline(study_xgb.best_value, color='red', linestyle='--', label=f'Best: {study_xgb.best_value:.4f}')
    ax.axhline(xgb_score.mean(), color='gray', linestyle=':', label=f'Default: {xgb_score.mean():.4f}')
    ax.set_title('XGBoost Optimization History', fontweight='bold')
    ax.set_xlabel('Trial'); ax.set_ylabel('Accuracy'); ax.legend()
    
    ax = axes[1]
    importances = optuna.importance.get_param_importances(study_xgb)
    ax.barh(list(importances.keys()), list(importances.values()), color='#E74C3C', alpha=0.7)
    ax.set_title('XGBoost Parameter Importance', fontweight='bold')
    ax.set_xlabel('Importance')
    
    plt.tight_layout(); plt.show()

---

## Part 4: Comparison — Default vs Tuned

In [ ]:
# ============================================================
# 4.1 Summary comparison
# ============================================================
print('\nMODEL COMPARISON: DEFAULT vs TUNED')
print('=' * 60)
print(f'{"Model":25s} | {"Default":>8s} | {"Tuned":>8s} | {"Δ":>8s}')
print('-' * 60)
print(f'{"Random Forest":25s} | {rf_score.mean():8.4f} | {rf_best_score:8.4f} | {(rf_best_score-rf_score.mean())*100:+7.2f}%')
if XGB_OK:
    print(f'{"XGBoost":25s} | {xgb_score.mean():8.4f} | {xgb_best_score:8.4f} | {(xgb_best_score-xgb_score.mean())*100:+7.2f}%')

In [ ]:
# ============================================================
# 4.2 Visual comparison
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

models_comp = ['Random Forest']
default_scores = [rf_score.mean()]
tuned_scores = [rf_best_score]

if XGB_OK:
    models_comp.append('XGBoost')
    default_scores.append(xgb_score.mean())
    tuned_scores.append(xgb_best_score)

x = np.arange(len(models_comp))
width = 0.35

ax.bar(x - width/2, default_scores, width, label='Default', color='#BDC3C7', alpha=0.8)
ax.bar(x + width/2, tuned_scores, width, label='Tuned', color='#2ECC71', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(models_comp)
ax.set_ylabel('Accuracy')
ax.set_title('Default vs Tuned Performance', fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Random')
ax.set_ylim(0.45, max(max(default_scores), max(tuned_scores)) + 0.05)

# Add value labels
for i, (d, t) in enumerate(zip(default_scores, tuned_scores)):
    ax.text(i - width/2, d + 0.005, f'{d:.3f}', ha='center', va='bottom', fontsize=10)
    ax.text(i + width/2, t + 0.005, f'{t:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout(); plt.show()

---

## Part 5: Learning Curve Analysis

Do we need more data, or would more features/complexity help?

In [ ]:
# ============================================================
# 5.1 Learning curve
# ============================================================
from sklearn.model_selection import learning_curve

rf_tuned = RandomForestClassifier(**rf_best_params, random_state=42, n_jobs=-1)

train_sizes, train_scores, val_scores = learning_curve(
    rf_tuned, X, y, cv=tscv,
    train_sizes=np.linspace(0.2, 1.0, 8),
    scoring='accuracy', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#3498DB', label='Training', linewidth=2)
ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
               train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='#3498DB')
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#E74C3C', label='Validation', linewidth=2)
ax.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
               val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='#E74C3C')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Learning Curve — Tuned Random Forest', fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Training accuracy:   {train_scores.mean(axis=1)[-1]:.4f}')
print(f'Validation accuracy: {val_scores.mean(axis=1)[-1]:.4f}')
gap = train_scores.mean(axis=1)[-1] - val_scores.mean(axis=1)[-1]
print(f'Gap: {gap:.4f} ({"High variance (overfitting)" if gap > 0.1 else "Good generalization"})')

**Learning Curve Interpretation:**
- If training >> validation: model is overfitting → need more data or regularization
- If both converge low: model is underfitting → need more features or complexity
- Large gap at all sizes → fundamental overfitting issue

In [ ]:
# ============================================================
# Save best parameters
# ============================================================
best_results = {
    'rf_best_accuracy': rf_best_score,
    'rf_params': str(rf_best_params)
}
if XGB_OK:
    best_results['xgb_best_accuracy'] = xgb_best_score
    best_results['xgb_params'] = str(xgb_best_params)

pd.Series(best_results).to_csv(os.path.join(PROCESSED_DIR, 'tuning_results.csv'))
print('✅ Saved: tuning_results.csv')

---

## 📌 Phase 4 Insight: The "Sharpe" of Your Model

> **The 50-Year Veteran says:** *"Backtests are just marketing brochures. Show me the pain."*  
> **The Data Scientist says:** *"Time series cross-validation is non-negotiable."*

### Beyond Accuracy: The Model Sharpe Ratio
Don't just check **Accuracy (%)**. Check the **Sharpe Ratio** of the strategy the model suggests.

$$\text{Sharpe Ratio} = \frac{R_p - R_f}{\sigma_p}$$

Where $R_p$ = portfolio return, $R_f$ = risk-free rate, $\sigma_p$ = portfolio standard deviation.

**Why this matters:** A model that is 60% accurate but loses huge money when it's wrong is **useless**. The Sharpe Ratio captures both the return AND the risk of the strategy — a Sharpe above 1.0 is good, above 2.0 is excellent.

### Walk-Forward Validation (Expanding Window)
Never use standard K-Fold cross-validation for time series. You cannot train on 2024 data to predict 2023:

```
Train: 2018-2020 → Test: Q1 2021
Train: 2018-Q1 2021 → Test: Q2 2021
Train: 2018-Q2 2021 → Test: Q3 2021
```

> **The test set must ALWAYS come after the training set chronologically.**


---

## Summary

### 🔑 Key Findings:

1. **Tuning improves accuracy by 1-3%** — seems small but compounds over thousands of trades
2. **n_estimators and max_depth** are the most impactful RF parameters
3. **learning_rate** is the most critical XGBoost parameter
4. **Optuna's Bayesian approach** finds good parameters in fewer trials than grid search
5. **Learning curves** show whether more data or more complexity would help

### The Diminishing Returns of Tuning
Beyond a certain point, hyperparameter tuning yields diminishing returns. The biggest improvements come from:
1. Better features (NB09)
2. Better data (more history, alternative data)
3. Better problem formulation (weekly vs daily prediction)

---
*Next: Notebook 11 — Forecasting with Prophet*